# trainer-subclass-extend — faded example 1: Subclass _step and add one metric key via super()

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `trainer-subclass-extend`. Running the beacon reports progress on the `Trainer: subclass extend pattern` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Trainer: subclass extend pattern` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`trainer-subclass-extend`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "trainer-subclass-extend"
DD_SUBTOPIC = "Trainer: subclass extend pattern"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

The subclass-extend pattern for `_step`: call `super()._step(batch)` to get the base dict, then add new keys to it and return. The MRO ensures `super()` calls the nearest parent in the chain, not necessarily the topmost base class.

## Faded exercise 1

A `BaseTrainer` is provided with `_step(batch)` returning `{'loss': float(batch[0].abs().sum())}`. Implement `GradNormTrainer(BaseTrainer)` whose `_step(batch)` calls `super()._step(batch)`, then adds `'batch_mean': float(batch[0].mean())` to the dict. The blank is the `super()._step(batch)` delegation call that runs the base step first.

**Fill in:** The super()._step(batch) call that runs the base training step and returns the initial dict.

In [ ]:
import torch as t
import torch.nn as nn

class BaseTrainer:
    def __init__(self, model, lr=1e-2):
        self.model = model
        self.opt = t.optim.SGD(model.parameters(), lr=lr)
        self.loss_fn = nn.MSELoss()

    def _step(self, batch):
        x, y = batch
        pred = self.model(x)
        loss = self.loss_fn(pred, y)
        self.opt.zero_grad()
        loss.backward()
        self.opt.step()
        return {'loss': loss.item()}

class GradNormTrainer(BaseTrainer):
    def _step(self, batch):
        raise NotImplementedError()  # TODO: The super()._step(batch) call that runs the base training step and returns the initial dict.
        out['batch_mean'] = float(batch[0].mean())
        return out


def _test():
    import torch as t
    import torch.nn as nn
    t.manual_seed(0)
    model = nn.Linear(3, 1)
    trainer = GradNormTrainer(model, lr=0.01)
    batch = (t.randn(4, 3), t.randn(4, 1))
    result = trainer._step(batch)
    assert 'loss' in result, 'base loss key missing'
    assert 'batch_mean' in result, 'extended batch_mean key missing'
    assert isinstance(result['loss'], float)
    assert isinstance(result['batch_mean'], float)
    # batch_mean should match independent ground truth
    expected_mean = float(batch[0].mean())
    assert abs(result['batch_mean'] - expected_mean) < 1e-5


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t
import torch.nn as nn

class BaseTrainer:
    def __init__(self, model, lr=1e-2):
        self.model = model
        self.opt = t.optim.SGD(model.parameters(), lr=lr)
        self.loss_fn = nn.MSELoss()

    def _step(self, batch):
        x, y = batch
        pred = self.model(x)
        loss = self.loss_fn(pred, y)
        self.opt.zero_grad()
        loss.backward()
        self.opt.step()
        return {'loss': loss.item()}

class GradNormTrainer(BaseTrainer):
    def _step(self, batch):
        out = super()._step(batch)
        out['batch_mean'] = float(batch[0].mean())
        return out
```
</details>